# Spotify Agent Playground

This notebook lets you test the Spotify music agent directly.

It uses:
- Spotify Web API via the local project client
- watsonx `openai/gpt-oss-120b` for response shaping
- the same `answer_music_question()` path used by the current music agent


## Setup

Before running this notebook, make sure the Jupyter kernel is using a Python environment where this project is installed, for example:

`<your-venv-path>/bin/python`

or a local project environment such as:

`./.venv/bin/python`

This notebook reads credentials from the repo `.env` file.

Also start the Spotify MCP server first in a terminal:

`source <your-venv-path>/bin/activate && python -m a2a_music_concert.spotify_mcp.server`


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    for candidate in [Path.cwd().parent, *Path.cwd().parents]:
        if (candidate / "src").exists():
            ROOT = candidate
            break

if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

print("Project source path configured.")

In [ ]:
from dotenv import load_dotenv

load_dotenv(ROOT / ".env")
print("Loaded .env configuration.")

In [ ]:
import asyncio
import json

from a2a_music_concert.music_agent.service import answer_music_question
from a2a_music_concert.shared.config import get_settings
from a2a_music_concert.spotify_mcp.spotify_client import SpotifyClient

settings = get_settings()
spotify_client = SpotifyClient(settings)

print("watsonx model:", settings.watsonx_model_id)
print("spotify redirect uri:", settings.spotify_redirect_uri)
print("music agent port:", settings.music_agent_port)

## Inspect Raw Spotify Data

Run this cell if you want to see what the agent is using before it asks the model to shape the answer.

In [ ]:
async def inspect_spotify():
    top_artists = [artist.model_dump() for artist in await spotify_client.get_top_artists(limit=5)]
    recent_tracks = [track.model_dump() for track in await spotify_client.get_recently_played(limit=10)]
    return {
        "top_artists": top_artists,
        "recent_tracks": recent_tracks,
    }

spotify_snapshot = await inspect_spotify()
print(json.dumps(spotify_snapshot, indent=2))

## Query Playground

Change the `query` string and rerun the cell.

In [ ]:
query = "Whos my most played song of all time"

result = await answer_music_question(query)
result.model_dump()

## Batch Test

Edit the list below to compare several prompts quickly.

In [ ]:
queries = [
    "Who is my top artist recently?",
    "Who have I been listening to most lately?",
    "What artist am I into the most right now?",
]

results = []
for q in queries:
    response = await answer_music_question(q)
    results.append({"query": q, "response": response.model_dump()})

print(json.dumps(results, indent=2))